In [ ]:
%pip install -q streamlit pyngrok PyJWT bcrypt


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 122.2 MB/s eta 0:00:00


In [6]:
from google.colab import userdata

required_secrets = [
    "JWT_SECRET",
    "NGROK_AUTHTOKEN",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD",
    "ADMIN_EMAIL",
    "ADMIN_PASSWORD",
]

secret_values = {}

for name in required_secrets:
    try:
        value = userdata.get(name)
        if not value or not value.strip():
            print(f"❌ {name}: Empty or missing")
        else:
            secret_values[name] = value
            print(f"✅ {name}: Available")
    except Exception:
        print(f"❌ {name}: Not accessible — check Colab Secrets")

if len(secret_values) == len(required_secrets):
    if len(secret_values["JWT_SECRET"]) < 32:
        print("\n⚠️ JWT_SECRET should contain at least 32 characters.")
    elif secret_values["EMAIL_ADDRESS"].strip().lower() == \
            secret_values["ADMIN_EMAIL"].strip().lower():
        print("\n⚠️ Admin and sender emails are the same. "
              "This is allowed, but separate accounts are preferable.")
    else:
        print("\n✅ Secret configuration looks good.")

✅ JWT_SECRET: Available
✅ NGROK_AUTHTOKEN: Available
✅ EMAIL_ADDRESS: Available
✅ EMAIL_PASSWORD: Available
✅ ADMIN_EMAIL: Available
✅ ADMIN_PASSWORD: Available

✅ Secret configuration looks good.


In [7]:
import sqlite3
import bcrypt
import re
from pathlib import Path
from datetime import datetime, timezone


In [9]:
# Database location in the current Google Colab session
DB_PATH = "/content/infosys_portal.db"


def get_db_connection():
    """Create a connection to the shared SQLite database."""
    conn = sqlite3.connect(DB_PATH, timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_db():
    """Create the tables required for authentication and OTP recovery."""
    with get_db_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL COLLATE NOCASE UNIQUE,
                email TEXT NOT NULL COLLATE NOCASE UNIQUE,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS otp_codes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL COLLATE NOCASE,
                otp_hash TEXT NOT NULL,
                purpose TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                used INTEGER NOT NULL DEFAULT 0,
                created_at TEXT NOT NULL
            )
        """)

        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_otp_email_purpose
            ON otp_codes(email, purpose)
        """)

    print("✅ Database and tables initialized successfully.")



In [10]:
def hash_password(password):
    """Hash a password before storing it."""
    if not isinstance(password, str) or not password:
        raise ValueError("Password cannot be empty.")

    # bcrypt has a 72-byte input limit.
    if len(password.encode("utf-8")) > 72:
        raise ValueError("Password must be no longer than 72 UTF-8 bytes.")

    return bcrypt.hashpw(
        password.encode("utf-8"),
        bcrypt.gensalt(rounds=12)
    ).decode("utf-8")


def verify_password(password, stored_hash):
    """Verify a password against its stored bcrypt hash."""
    try:
        if len(password.encode("utf-8")) > 72:
            return False

        return bcrypt.checkpw(
            password.encode("utf-8"),
            stored_hash.encode("utf-8")
        )
    except (ValueError, AttributeError, TypeError):
        return False


def hash_security_answer(answer):
    """Normalize and hash a security-question answer."""
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("Security answer cannot be empty.")

    normalized = answer.strip().casefold()
    return hash_password(normalized)


def verify_security_answer(answer, stored_hash):
    """Verify a normalized security-question answer."""
    if not isinstance(answer, str):
        return False

    try:
        return verify_password(answer.strip().casefold(), stored_hash)
    except ValueError:
        return False


def validate_email(email):
    """Basic email-format validation."""
    if not isinstance(email, str):
        return False

    email = email.strip()
    return bool(
        re.fullmatch(
            r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
            email
        )
    )


# Initialize the database
init_db()

# Test password hashing without storing any real credentials
test_password = "MilestoneTest123!"
test_hash = hash_password(test_password)

assert verify_password(test_password, test_hash)
assert not verify_password("WrongPassword123!", test_hash)
assert validate_email("student@example.com")
assert not validate_email("not-an-email")

print("✅ Password hashing test passed.")
print("✅ Incorrect-password rejection test passed.")
print("✅ Email validation tests passed.")
print("Database path:", DB_PATH)


✅ Database and tables initialized successfully.
✅ Password hashing test passed.
✅ Incorrect-password rejection test passed.
✅ Email validation tests passed.
Database path: /content/infosys_portal.db


In [11]:
import re
import sqlite3
from datetime import datetime, timezone


In [12]:
def register_user(username, email, password, confirm_password,
                 security_question, security_answer):
    """Validate and register a new user in the shared SQLite database."""

    # 1. Normalize user inputs
    username = username.strip() if isinstance(username, str) else ""
    email = email.strip().lower() if isinstance(email, str) else ""
    security_question = (
        security_question.strip()
        if isinstance(security_question, str) else ""
    )

    # 2. Validate username
    if not re.fullmatch(r"[A-Za-z0-9_]{3,30}", username):
        return False, (
            "Username must be 3–30 characters and contain only "
            "letters, numbers, and underscores."
        )

    # 3. Validate email
    if not validate_email(email):
        return False, "Please enter a valid email address."

    # 4. Validate password and confirmation
    if not isinstance(password, str) or not password:
        return False, "Password is required."

    if len(password) < 8:
        return False, "Password must contain at least 8 characters."

    if len(password.encode("utf-8")) > 72:
        return False, "Password must be no longer than 72 UTF-8 bytes."

    if not re.search(r"[A-Z]", password):
        return False, "Password must contain at least one uppercase letter."

    if not re.search(r"[a-z]", password):
        return False, "Password must contain at least one lowercase letter."

    if not re.search(r"\d", password):
        return False, "Password must contain at least one digit."

    if not re.search(r"[^A-Za-z0-9]", password):
        return False, "Password must contain at least one special character."

    if password != confirm_password:
        return False, "Password and confirmation do not match."

    # 5. Validate security question and answer
    if not security_question:
        return False, "Please select or enter a security question."

    if not isinstance(security_answer, str) or not security_answer.strip():
        return False, "Security answer is required."

    try:
        # Hash values before storing them
        password_hash = hash_password(password)
        answer_hash = hash_security_answer(security_answer)

        with get_db_connection() as conn:
            # Check for existing username or email
            existing = conn.execute(
                """
                SELECT username, email
                FROM users
                WHERE username = ? COLLATE NOCASE
                   OR email = ? COLLATE NOCASE
                LIMIT 1
                """,
                (username, email)
            ).fetchone()

            if existing:
                if existing["username"].lower() == username.lower():
                    return False, "This username is already registered."
                return False, "This email address is already registered."

            conn.execute(
                """
                INSERT INTO users (
                    username,
                    email,
                    password_hash,
                    security_question,
                    security_answer_hash,
                    created_at
                )
                VALUES (?, ?, ?, ?, ?, ?)
                """,
                (
                    username,
                    email,
                    password_hash,
                    security_question,
                    answer_hash,
                    datetime.now(timezone.utc).isoformat()
                )
            )

        return True, "Registration successful. You can now log in."

    except sqlite3.IntegrityError:
        # Handles duplicates even if two requests arrive simultaneously
        return False, "Username or email is already registered."

    except Exception:
        # Avoid exposing internal database details to the user
        return False, "Registration could not be completed. Please try again."


In [13]:

import uuid

# Generate a unique temporary test account
test_id = uuid.uuid4().hex[:8]
test_username = f"test_{test_id}"
test_email = f"test_{test_id}@example.com"
test_password = "TestPass123!"

try:
    # Test 1: Successful registration
    success, message = register_user(
        username=test_username,
        email=test_email,
        password=test_password,
        confirm_password=test_password,
        security_question="What is your favourite colour?",
        security_answer="Blue"
    )

    assert success, f"Registration test failed: {message}"
    print("✅ Valid user registration passed.")

    # Test 2: Duplicate username/email is rejected
    duplicate_success, duplicate_message = register_user(
        username=test_username,
        email=test_email,
        password=test_password,
        confirm_password=test_password,
        security_question="What is your favourite colour?",
        security_answer="Blue"
    )

    assert not duplicate_success
    print("✅ Duplicate-account prevention passed.")

    # Test 3: Incorrect password confirmation is rejected
    mismatch_success, mismatch_message = register_user(
        username=f"other_{test_id}",
        email=f"other_{test_id}@example.com",
        password=test_password,
        confirm_password="DifferentPass123!",
        security_question="What is your favourite colour?",
        security_answer="Blue"
    )

    assert not mismatch_success
    print("✅ Password-confirmation validation passed.")

    # Test 4: Check that the password was stored as a hash
    with get_db_connection() as conn:
        row = conn.execute(
            "SELECT password_hash FROM users WHERE username = ?",
            (test_username,)
        ).fetchone()

    assert row is not None
    assert row["password_hash"] != test_password
    assert row["password_hash"].startswith("$2")
    assert verify_password(test_password, row["password_hash"])
    print("✅ Password hashing and verification passed.")

finally:
    # Remove only the temporary test account
    with get_db_connection() as conn:
        conn.execute(
            "DELETE FROM users WHERE username = ? AND email = ?",
            (test_username, test_email)
        )

    print("🧹 Temporary test account cleaned up.")


✅ Valid user registration passed.
✅ Duplicate-account prevention passed.
✅ Password-confirmation validation passed.
✅ Password hashing and verification passed.
🧹 Temporary test account cleaned up.


In [14]:

import jwt
from datetime import datetime, timezone, timedelta
from google.colab import userdata


In [15]:
# Read the JWT signing secret securely from Colab Secrets
JWT_SECRET = userdata.get("JWT_SECRET")
JWT_ALGORITHM = "HS256"
JWT_EXPIRY_MINUTES = 30

In [16]:
def authenticate_user(identifier, password):
    """Authenticate using username or email."""
    if not isinstance(identifier, str) or not isinstance(password, str):
        return False, "Invalid username/email or password.", None

    identifier = identifier.strip()

    if not identifier or not password:
        return False, "Invalid username/email or password.", None

    try:
        with get_db_connection() as conn:
            user = conn.execute(
                """
                SELECT id, username, email, password_hash
                FROM users
                WHERE username = ? COLLATE NOCASE
                   OR email = ? COLLATE NOCASE
                LIMIT 1
                """,
                (identifier, identifier)
            ).fetchone()

        # Use the same message for unknown users and incorrect passwords
        # to avoid revealing which accounts exist.
        if user is None or not verify_password(
            password, user["password_hash"]
        ):
            return False, "Invalid username/email or password.", None

        token = make_jwt(user["id"], user["username"], user["email"])
        return True, "Login successful.", token

    except Exception:
        return False, "Login could not be completed. Please try again.", None


def make_jwt(user_id, username, email):
    """Create a signed JWT with an expiration time."""
    now = datetime.now(timezone.utc)

    payload = {
        "sub": str(user_id),
        "username": username,
        "email": email,
        "iat": now,
        "exp": now + timedelta(minutes=JWT_EXPIRY_MINUTES)
    }

    return jwt.encode(payload, JWT_SECRET, algorithm=JWT_ALGORITHM)


def verify_jwt(token):
    """Validate a JWT and return its claims, or None if invalid."""
    if not isinstance(token, str) or not token:
        return None

    try:
        return jwt.decode(
            token,
            JWT_SECRET,
            algorithms=[JWT_ALGORITHM],
            options={"require": ["sub", "iat", "exp"]}
        )
    except jwt.InvalidTokenError:
        return None


In [17]:

import uuid

test_id = uuid.uuid4().hex[:8]
test_username = f"login_{test_id}"
test_email = f"login_{test_id}@example.com"
test_password = "LoginTest123!"

try:
    # Create a temporary user
    ok, msg = register_user(
        username=test_username,
        email=test_email,
        password=test_password,
        confirm_password=test_password,
        security_question="What is your favourite colour?",
        security_answer="Blue"
    )
    assert ok, f"Test account creation failed: {msg}"
    print("✅ Temporary test user created.")

    # Test login using username
    ok, msg, token = authenticate_user(test_username, test_password)
    assert ok and token, f"Username login failed: {msg}"
    print("✅ Login with username passed.")

    # Test login using email
    ok, msg, email_token = authenticate_user(test_email, test_password)
    assert ok and email_token, f"Email login failed: {msg}"
    print("✅ Login with email passed.")

    # Test incorrect password
    ok, msg, bad_token = authenticate_user(
        test_username, "WrongPassword123!"
    )
    assert not ok and bad_token is None
    print("✅ Incorrect password rejected.")

    # Test the JWT generated after successful login
    claims = verify_jwt(token)
    assert claims is not None
    assert claims["username"] == test_username
    assert claims["email"] == test_email
    print("✅ JWT signature and claims validated.")

    # Test a malformed token
    assert verify_jwt("not-a-valid-token") is None
    print("✅ Invalid JWT rejected.")

    # Test expiration enforcement with a deliberately expired token
    expired_payload = {
        "sub": "test-user",
        "iat": datetime.now(timezone.utc) - timedelta(minutes=10),
        "exp": datetime.now(timezone.utc) - timedelta(minutes=5)
    }
    expired_token = jwt.encode(
        expired_payload, JWT_SECRET, algorithm=JWT_ALGORITHM
    )
    assert verify_jwt(expired_token) is None
    print("✅ Expired JWT rejected.")

finally:
    with get_db_connection() as conn:
        conn.execute(
            "DELETE FROM users WHERE username = ? AND email = ?",
            (test_username, test_email)
        )

    print("🧹 Temporary test user cleaned up.")


✅ Temporary test user created.
✅ Login with username passed.
✅ Login with email passed.
✅ Incorrect password rejected.
✅ JWT signature and claims validated.
✅ Invalid JWT rejected.
✅ Expired JWT rejected.
🧹 Temporary test user cleaned up.


In [18]:
import os
from google.colab import userdata

for name in [
    "JWT_SECRET",
    "EMAIL_ADDRESS",
    "EMAIL_PASSWORD",
    "ADMIN_EMAIL",
    "ADMIN_PASSWORD",
    "NGROK_AUTHTOKEN",
]:
    os.environ[name] = userdata.get(name)

print("✅ Application configuration loaded securely.")

✅ Application configuration loaded securely.


In [19]:

%%writefile app.py
import os
import re
import hmac
import sqlite3
from datetime import datetime, timezone, timedelta

import bcrypt
import jwt
import streamlit as st

# ---------- Configuration ----------
DB_PATH = "/content/infosys_portal.db"
JWT_SECRET = os.environ["JWT_SECRET"]
JWT_ALGORITHM = "HS256"
JWT_EXPIRY_MINUTES = 30

ADMIN_EMAIL = os.environ["ADMIN_EMAIL"].strip().lower()
ADMIN_PASSWORD = os.environ["ADMIN_PASSWORD"]


# ---------- Database ----------
def get_db_connection():
    conn = sqlite3.connect(DB_PATH, timeout=10)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_db():
    with get_db_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL COLLATE NOCASE UNIQUE,
                email TEXT NOT NULL COLLATE NOCASE UNIQUE,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS otp_codes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL COLLATE NOCASE,
                otp_hash TEXT NOT NULL,
                purpose TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                used INTEGER NOT NULL DEFAULT 0,
                created_at TEXT NOT NULL
            )
        """)


# ---------- Security ----------
def hash_password(password):
    if len(password.encode("utf-8")) > 72:
        raise ValueError("Password must be at most 72 UTF-8 bytes.")
    return bcrypt.hashpw(
        password.encode("utf-8"), bcrypt.gensalt(rounds=12)
    ).decode("utf-8")


def verify_password(password, password_hash):
    try:
        if len(password.encode("utf-8")) > 72:
            return False
        return bcrypt.checkpw(
            password.encode("utf-8"), password_hash.encode("utf-8")
        )
    except (ValueError, TypeError, AttributeError):
        return False


def hash_answer(answer):
    return hash_password(answer.strip().casefold())


def valid_email(email):
    return bool(re.fullmatch(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        email.strip()
    ))


def valid_password(password):
    return (
        len(password) >= 8
        and len(password.encode("utf-8")) <= 72
        and re.search(r"[A-Z]", password)
        and re.search(r"[a-z]", password)
        and re.search(r"\d", password)
        and re.search(r"[^A-Za-z0-9]", password)
    )


# ---------- Registration ----------
def register_user(username, email, password, confirmation,
                  question, answer):
    username = username.strip()
    email = email.strip().lower()
    question = question.strip()

    if not re.fullmatch(r"[A-Za-z0-9_]{3,30}", username):
        return False, "Username must be 3–30 letters, numbers, or underscores."

    if not valid_email(email):
        return False, "Enter a valid email address."

    if not valid_password(password):
        return False, (
            "Password needs 8+ characters, uppercase and lowercase letters, "
            "a digit, and a special character; maximum 72 UTF-8 bytes."
        )

    if password != confirmation:
        return False, "Passwords do not match."

    if not question or not answer.strip():
        return False, "Select a security question and enter an answer."

    try:
        with get_db_connection() as conn:
            conn.execute("""
                INSERT INTO users (
                    username, email, password_hash, security_question,
                    security_answer_hash, created_at
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (
                username,
                email,
                hash_password(password),
                question,
                hash_answer(answer),
                datetime.now(timezone.utc).isoformat()
            ))
        return True, "Account created successfully. You can now log in."

    except sqlite3.IntegrityError:
        return False, "Username or email is already registered."


# ---------- Login and JWT ----------
def make_jwt(user):
    now = datetime.now(timezone.utc)
    return jwt.encode({
        "sub": str(user["id"]),
        "username": user["username"],
        "email": user["email"],
        "iat": now,
        "exp": now + timedelta(minutes=JWT_EXPIRY_MINUTES)
    }, JWT_SECRET, algorithm=JWT_ALGORITHM)


def verify_jwt(token):
    try:
        return jwt.decode(
            token,
            JWT_SECRET,
            algorithms=[JWT_ALGORITHM],
            options={"require": ["sub", "iat", "exp"]}
        )
    except (jwt.InvalidTokenError, TypeError):
        return None


def authenticate_user(identifier, password):
    identifier = identifier.strip()

    with get_db_connection() as conn:
        user = conn.execute("""
            SELECT id, username, email, password_hash
            FROM users
            WHERE username = ? COLLATE NOCASE
               OR email = ? COLLATE NOCASE
            LIMIT 1
        """, (identifier, identifier)).fetchone()

    if not user or not verify_password(password, user["password_hash"]):
        return None

    return make_jwt(user)


# ---------- Session helpers ----------
def logout():
    for key in ["auth_token", "admin_logged_in"]:
        st.session_state.pop(key, None)
    st.rerun()


# ---------- User interface ----------
st.set_page_config(
    page_title="Infosys Authentication Portal",
    page_icon="🔐",
    layout="centered"
)

init_db()

st.title("🔐 User Authentication System")
st.caption("Infosys Springboard 7.0 — Milestone 1")

# If a user token exists, validate it on every rerun.
token = st.session_state.get("auth_token")
claims = verify_jwt(token) if token else None

if token and not claims:
    st.session_state.pop("auth_token", None)
    st.warning("Your session has expired or is invalid. Please log in.")
    st.rerun()

if claims:
    st.success(f"Welcome, {claims['username']}!")
    st.subheader("User Dashboard")
    st.write("You are logged in.")
    st.write("**Username:**", claims["username"])
    st.write("**Email:**", claims["email"])
    st.caption("Your session expires after 30 minutes.")
    if st.button("Logout", type="primary"):
        logout()

elif st.session_state.get("admin_logged_in"):
    st.subheader("Administrator Dashboard")
    st.write("Registered user accounts")

    with get_db_connection() as conn:
        users = conn.execute("""
            SELECT id, username, email, created_at
            FROM users
            ORDER BY id DESC
        """).fetchall()

    if users:
        st.dataframe(
            [dict(user) for user in users],
            use_container_width=True,
            hide_index=True
        )
    else:
        st.info("No registered users yet.")

    st.caption("Password hashes and security answers are not displayed.")
    if st.button("Admin Logout", type="primary"):
        logout()

else:
    tab_register, tab_login, tab_admin = st.tabs([
        "Register", "User Login", "Admin Login"
    ])

    with tab_register:
        st.subheader("Create an Account")

        with st.form("register_form"):
            username = st.text_input("Username")
            email = st.text_input("Email address")
            password = st.text_input("Password", type="password")
            confirmation = st.text_input(
                "Confirm password", type="password"
            )
            question = st.selectbox(
                "Security question",
                [
                    "What is your favourite colour?",
                    "What was the name of your first school?",
                    "What city were you born in?"
                ]
            )
            answer = st.text_input("Security answer", type="password")
            submitted = st.form_submit_button(
                "Register", type="primary", use_container_width=True
            )

        if submitted:
            ok, message = register_user(
                username, email, password, confirmation, question, answer
            )
            if ok:
                st.success(message)
            else:
                st.error(message)

    with tab_login:
        st.subheader("User Login")

        with st.form("login_form"):
            identifier = st.text_input("Username or email")
            login_password = st.text_input("Password", type="password")
            login_submitted = st.form_submit_button(
                "Login", type="primary", use_container_width=True
            )

        if login_submitted:
            token = authenticate_user(identifier, login_password)
            if token:
                st.session_state["auth_token"] = token
                st.rerun()
            else:
                st.error("Invalid username/email or password.")

        st.info(
            "Forgot password / email OTP recovery will be added in a "
            "subsequent step."
        )

    with tab_admin:
        st.subheader("Administrator Login")

        with st.form("admin_login_form"):
            admin_email = st.text_input("Admin email")
            admin_password = st.text_input(
                "Admin password", type="password"
            )
            admin_submitted = st.form_submit_button(
                "Admin Login", type="primary", use_container_width=True
            )

        if admin_submitted:
            email_ok = hmac.compare_digest(
                admin_email.strip().lower(), ADMIN_EMAIL
            )
            password_ok = hmac.compare_digest(
                admin_password, ADMIN_PASSWORD
            )

            if email_ok and password_ok:
                st.session_state["admin_logged_in"] = True
                st.rerun()
            else:
                st.error("Invalid administrator credentials.")

st.divider()
st.caption("Authentication demo — do not use real or reused passwords.")


Writing app.py


In [20]:
from pathlib import Path

app_path = Path("/content/app.py")

assert app_path.exists(), "app.py was not created."
print("✅ app.py created successfully.")
print("File size:", app_path.stat().st_size, "bytes")

✅ app.py created successfully.
File size: 10166 bytes


In [21]:
import subprocess
import time
import requests

# Stop an older Streamlit process from this notebook, if any.
if "streamlit_process" in globals():
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=5)

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )
        if response.status_code == 200:
            print("✅ Streamlit server is running on port 8501.")
            break
    except requests.RequestException:
        pass

    time.sleep(1)
else:
    raise RuntimeError(
        "Streamlit did not start. Check app.py and package installation."
    )

✅ Streamlit server is running on port 8501.


In [33]:

from pyngrok import ngrok
from google.colab import userdata

# Reuse the existing ngrok configuration.
# Only set the token if ngrok has no saved configuration.
try:
    ngrok.get_ngrok_process()
    print("An ngrok process is already running.")
except Exception:
    token = userdata.get("NGROK_AUTHTOKEN")

    if not token:
        raise ValueError(
            "NGROK_AUTHTOKEN is missing from Colab Secrets."
        )

    try:
        ngrok.set_auth_token(token)
        print("Ngrok authentication configured.")
    except Exception as e:
        if "Authtoken already set in configuration file" in str(e):
            print("Ngrok authtoken is already configured.")
        else:
            raise


An ngrok process is already running.


In [23]:

from pathlib import Path

app_file = Path("/content/app.py")
source = app_file.read_text(encoding="utf-8")

if "def request_password_reset(email):" in source:
    raise RuntimeError("OTP code is already present in app.py.")

otp_functions = r'''
# ---------- Email OTP password recovery ----------
def request_password_reset(email):
    import secrets
    import smtplib
    from email.message import EmailMessage

    email = email.strip().lower()

    with get_db_connection() as conn:
        user = conn.execute(
            "SELECT id FROM users WHERE email = ? COLLATE NOCASE",
            (email,)
        ).fetchone()

    if not user:
        return False

    otp = f"{secrets.randbelow(1_000_000):06d}"
    now = datetime.now(timezone.utc)
    expires = now + timedelta(minutes=5)

    with get_db_connection() as conn:
        conn.execute("""
            UPDATE otp_codes SET used = 1
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset' AND used = 0
        """, (email,))
        conn.execute("""
            INSERT INTO otp_codes (
                email, otp_hash, purpose, expires_at,
                attempts, used, created_at
            ) VALUES (?, ?, 'password_reset', ?, 0, 0, ?)
        """, (
            email, hash_password(otp), expires.isoformat(), now.isoformat()
        ))

    message = EmailMessage()
    message["Subject"] = "Your password reset verification code"
    message["From"] = os.environ["EMAIL_ADDRESS"]
    message["To"] = email
    message.set_content(
        f"Your password reset code is: {otp}\n\n"
        "It expires in 5 minutes and can be used only once. "
        "Ignore this email if you did not request a reset."
    )

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as server:
            server.login(
                os.environ["EMAIL_ADDRESS"],
                os.environ["EMAIL_PASSWORD"]
            )
            server.send_message(message)
        return True
    except Exception:
        with get_db_connection() as conn:
            conn.execute("""
                UPDATE otp_codes SET used = 1
                WHERE email = ? COLLATE NOCASE
                  AND purpose = 'password_reset' AND used = 0
            """, (email,))
        return False


def verify_otp_and_reset_password(email, otp, new_password, confirmation):
    email = email.strip().lower()

    if not valid_email(email):
        return False, "Enter a valid email address."
    if not re.fullmatch(r"\d{6}", otp.strip()):
        return False, "Enter the six-digit OTP."
    if not valid_password(new_password):
        return False, (
            "Password needs 8+ characters, uppercase and lowercase letters, "
            "a digit, and a special character; maximum 72 UTF-8 bytes."
        )
    if new_password != confirmation:
        return False, "Passwords do not match."

    now = datetime.now(timezone.utc)

    with get_db_connection() as conn:
        record = conn.execute("""
            SELECT id, otp_hash, expires_at, attempts
            FROM otp_codes
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset' AND used = 0
            ORDER BY id DESC LIMIT 1
        """, (email,)).fetchone()

        if record is None:
            return False, "OTP is invalid, expired, or already used."

        if record["attempts"] >= 5:
            conn.execute(
                "UPDATE otp_codes SET used = 1 WHERE id = ?",
                (record["id"],)
            )
            return False, "Too many OTP attempts. Request a new code."

        if datetime.fromisoformat(record["expires_at"]) <= now:
            conn.execute(
                "UPDATE otp_codes SET used = 1 WHERE id = ?",
                (record["id"],)
            )
            return False, "OTP has expired. Request a new code."

        conn.execute(
            "UPDATE otp_codes SET attempts = attempts + 1 WHERE id = ?",
            (record["id"],)
        )

        if not verify_password(otp.strip(), record["otp_hash"]):
            return False, "Incorrect OTP."

        user = conn.execute(
            "SELECT id FROM users WHERE email = ? COLLATE NOCASE",
            (email,)
        ).fetchone()

        if user is None:
            conn.execute(
                "UPDATE otp_codes SET used = 1 WHERE id = ?",
                (record["id"],)
            )
            return False, "Password reset could not be completed."

        conn.execute(
            "UPDATE users SET password_hash = ? WHERE id = ?",
            (hash_password(new_password), user["id"])
        )
        conn.execute("""
            UPDATE otp_codes SET used = 1
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset' AND used = 0
        """, (email,))

    return True, "Password reset successful. You can now log in."
'''

marker = "# ---------- Session helpers ----------"
old_tabs = '''    tab_register, tab_login, tab_admin = st.tabs([
        "Register", "User Login", "Admin Login"
    ])'''
old_placeholder = '''        st.info(
            "Forgot password / email OTP recovery will be added in a "
            "subsequent step."
        )

    with tab_admin:'''

new_tabs = '''    tab_register, tab_login, tab_forgot, tab_admin = st.tabs([
        "Register", "User Login", "Forgot Password", "Admin Login"
    ])'''

new_recovery_ui = '''
    with tab_forgot:
        st.subheader("Reset Password with Email OTP")
        st.write("Request a six-digit verification code sent to your email.")

        with st.form("request_otp_form"):
            reset_email_input = st.text_input("Registered email address")
            request_otp = st.form_submit_button(
                "Send OTP", use_container_width=True
            )

        if request_otp:
            if not valid_email(reset_email_input):
                st.error("Enter a valid email address.")
            else:
                sent = request_password_reset(reset_email_input)
                st.session_state["reset_email"] = (
                    reset_email_input.strip().lower()
                )
                if sent:
                    st.success(
                        "If this address is registered and email delivery "
                        "succeeded, a code has been sent. Check your inbox."
                    )
                else:
                    st.info(
                        "If the address is registered, check your inbox. "
                        "If no code arrives, verify the email configuration "
                        "and address, then try again."
                    )

        if st.session_state.get("reset_email"):
            st.caption("The OTP expires after 5 minutes.")
            with st.form("verify_otp_form"):
                entered_otp = st.text_input("Six-digit OTP", max_chars=6)
                new_password = st.text_input(
                    "New password", type="password"
                )
                confirm_new_password = st.text_input(
                    "Confirm new password", type="password"
                )
                reset_submitted = st.form_submit_button(
                    "Verify OTP and Reset Password",
                    use_container_width=True
                )

            if reset_submitted:
                ok, message = verify_otp_and_reset_password(
                    st.session_state["reset_email"],
                    entered_otp,
                    new_password,
                    confirm_new_password
                )
                if ok:
                    st.success(message)
                    st.session_state.pop("reset_email", None)
                else:
                    st.error(message)

    with tab_admin:'''

assert marker in source, "Session-helper marker not found."
assert old_tabs in source, "Existing tabs definition not found."
assert old_placeholder in source, "Existing OTP placeholder not found."

source = source.replace(marker, otp_functions + "\n\n" + marker, 1)
source = source.replace(old_tabs, new_tabs, 1)
source = source.replace(old_placeholder, new_recovery_ui, 1)

app_file.write_text(source, encoding="utf-8")

compile(source, str(app_file), "exec")
print("✅ OTP backend added.")
print("✅ Forgot Password tab added.")
print("✅ app.py syntax check passed.")
print("File size:", app_file.stat().st_size, "bytes")


✅ OTP backend added.
✅ Forgot Password tab added.
✅ app.py syntax check passed.
File size: 17050 bytes


In [24]:
import subprocess
import time
import requests

if "streamlit_process" in globals():
    if streamlit_process.poll() is None:
        streamlit_process.terminate()
        streamlit_process.wait(timeout=5)

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "/content/app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.headless", "true"
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

for _ in range(25):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health", timeout=2
        )
        if response.status_code == 200:
            print("✅ Updated Streamlit app is running.")
            break
    except requests.RequestException:
        pass
    time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start. Check app.py.")

✅ Updated Streamlit app is running.


In [25]:

# Step 1: Safe inspection of the current application
# This does NOT print passwords, tokens, or secret values.

import ast
import os
import sqlite3

APP_PATH = "/content/app.py"
DB_PATH = "/content/infosys_portal.db"

print("=== 1. Application file ===")
print("app.py exists:", os.path.exists(APP_PATH))

if os.path.exists(APP_PATH):
    with open(APP_PATH, "r", encoding="utf-8") as f:
        source = f.read()

    # Verify Python syntax
    ast.parse(source)
    print("Python syntax: OK")
    print("File size:", len(source), "characters")

    # List functions without displaying their code or secrets
    tree = ast.parse(source)
    functions = [
        node.name
        for node in ast.walk(tree)
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    ]
    print("\nFunctions found:")
    for name in sorted(set(functions)):
        print("-", name)
else:
    print("ERROR: app.py was not found.")

print("\n=== 2. Database structure ===")

if os.path.exists(DB_PATH):
    with sqlite3.connect(DB_PATH) as conn:
        tables = conn.execute("""
            SELECT name FROM sqlite_master
            WHERE type = 'table'
            ORDER BY name
        """).fetchall()

        for (table_name,) in tables:
            print(f"\nTable: {table_name}")
            columns = conn.execute(
                f'PRAGMA table_info("{table_name}")'
            ).fetchall()

            # Display column names and types only, never row data
            for column in columns:
                print(f"  - {column[1]}: {column[2]}")
else:
    print("Database file not found.")

print("\n=== 3. Security-related code markers ===")
if os.path.exists(APP_PATH):
    markers = [
        "request_password_reset",
        "verify_otp_and_reset_password",
        "expires_at",
        "attempts",
        "used",
        "jwt.decode",
        "verify_jwt",
        "st.rerun",
    ]

    for marker in markers:
        print(f"{marker}: {'Found' if marker in source else 'Not found'}")

print("\nInspection complete. No database records or secret values were printed.")


=== 1. Application file ===
app.py exists: True
Python syntax: OK
File size: 17038 characters

Functions found:
- authenticate_user
- get_db_connection
- hash_answer
- hash_password
- init_db
- logout
- make_jwt
- register_user
- request_password_reset
- valid_email
- valid_password
- verify_jwt
- verify_otp_and_reset_password
- verify_password

=== 2. Database structure ===

Table: otp_codes
  - id: INTEGER
  - email: TEXT
  - otp_hash: TEXT
  - purpose: TEXT
  - expires_at: TEXT
  - attempts: INTEGER
  - used: INTEGER
  - created_at: TEXT

Table: sqlite_sequence
  - name: 
  - seq: 

Table: users
  - id: INTEGER
  - username: TEXT
  - email: TEXT
  - password_hash: TEXT
  - security_question: TEXT
  - security_answer_hash: TEXT
  - created_at: TEXT

=== 3. Security-related code markers ===
request_password_reset: Found
verify_otp_and_reset_password: Found
expires_at: Found
attempts: Found
used: Found
jwt.decode: Found
verify_jwt: Found
st.rerun: Found

Inspection complete. No databas

In [27]:

import ast
import os
import shutil
from pathlib import Path

APP_PATH = Path("/content/app.py")
BACKUP_PATH = Path("/content/app.py.pre_security_patch.bak")
TEMP_PATH = Path("/content/app.py.security_tmp")

if not APP_PATH.exists():
    raise FileNotFoundError("app.py was not found. No changes made.")

if BACKUP_PATH.exists():
    raise FileExistsError(
        "A security-patch backup already exists. "
        "Keep it safe and rename it before running this cell again."
    )

original = APP_PATH.read_text(encoding="utf-8")
tree = ast.parse(original)

# Locate only the functions we intend to update.
targets = {
    "init_db",
    "make_jwt",
    "verify_jwt",
    "request_password_reset",
    "verify_otp_and_reset_password",
}
nodes = {
    node.name: node
    for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name in targets
}

missing = targets - set(nodes)
if missing:
    raise RuntimeError(f"Missing expected functions: {sorted(missing)}")

replacements = {}

replacements["init_db"] = '''
def init_db():
    with get_db_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL COLLATE NOCASE UNIQUE,
                email TEXT NOT NULL COLLATE NOCASE UNIQUE,
                password_hash TEXT NOT NULL,
                security_question TEXT NOT NULL,
                security_answer_hash TEXT NOT NULL,
                created_at TEXT NOT NULL,
                token_version INTEGER NOT NULL DEFAULT 0
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS otp_codes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL COLLATE NOCASE,
                otp_hash TEXT NOT NULL,
                purpose TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                used INTEGER NOT NULL DEFAULT 0,
                created_at TEXT NOT NULL
            )
        """)

        # Safe migration for an existing database.
        columns = {
            row["name"]
            for row in conn.execute("PRAGMA table_info(users)").fetchall()
        }
        if "token_version" not in columns:
            conn.execute("""
                ALTER TABLE users
                ADD COLUMN token_version INTEGER NOT NULL DEFAULT 0
            """)

        # Supports the per-email OTP request-limit query.
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_otp_email_purpose_created
            ON otp_codes(email, purpose, created_at)
        """)
'''

replacements["make_jwt"] = '''
def make_jwt(user):
    # Read the current token version from the database.
    with get_db_connection() as conn:
        current = conn.execute(
            "SELECT token_version FROM users WHERE id = ?",
            (user["id"],)
        ).fetchone()

    if current is None:
        raise ValueError("Cannot issue a token for a missing user.")

    now = datetime.now(timezone.utc)
    return jwt.encode({
        "sub": str(user["id"]),
        "username": user["username"],
        "email": user["email"],
        "ver": int(current["token_version"]),
        "iat": now,
        "exp": now + timedelta(minutes=JWT_EXPIRY_MINUTES)
    }, JWT_SECRET, algorithm=JWT_ALGORITHM)
'''

replacements["verify_jwt"] = '''
def verify_jwt(token):
    try:
        payload = jwt.decode(
            token,
            JWT_SECRET,
            algorithms=[JWT_ALGORITHM],
            options={"require": ["sub", "iat", "exp", "ver"]}
        )

        # Reject deleted users and tokens issued before a password reset.
        with get_db_connection() as conn:
            user = conn.execute(
                "SELECT token_version FROM users WHERE id = ?",
                (int(payload["sub"]),)
            ).fetchone()

        if user is None:
            return None

        if int(payload["ver"]) != int(user["token_version"]):
            return None

        return payload
    except (jwt.InvalidTokenError, TypeError, ValueError, KeyError):
        return None
'''

replacements["request_password_reset"] = '''
def request_password_reset(email):
    import secrets
    import smtplib
    from email.message import EmailMessage

    email = (email or "").strip().lower()
    if not valid_email(email):
        return False

    now = datetime.now(timezone.utc)
    cutoff = (now - timedelta(minutes=15)).isoformat()

    # Count all reset requests in the last 15 minutes, including used
    # or unsuccessfully delivered OTPs. Limit: 3 requests per email.
    with get_db_connection() as conn:
        user = conn.execute(
            "SELECT id FROM users WHERE email = ? COLLATE NOCASE",
            (email,)
        ).fetchone()

        if user is None:
            return False

        recent = conn.execute("""
            SELECT COUNT(*) AS total
            FROM otp_codes
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset'
              AND created_at >= ?
        """, (email, cutoff)).fetchone()["total"]

        if recent >= 3:
            return False

        otp = f"{secrets.randbelow(1_000_000):06d}"
        expires = now + timedelta(minutes=5)

        conn.execute("""
            UPDATE otp_codes SET used = 1
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset' AND used = 0
        """, (email,))

        conn.execute("""
            INSERT INTO otp_codes (
                email, otp_hash, purpose, expires_at,
                attempts, used, created_at
            ) VALUES (?, ?, 'password_reset', ?, 0, 0, ?)
        """, (
            email, hash_password(otp), expires.isoformat(), now.isoformat()
        ))

    message = EmailMessage()
    message["Subject"] = "Your password reset verification code"
    message["From"] = os.environ["EMAIL_ADDRESS"]
    message["To"] = email
    message.set_content(
        f"Your password reset code is: {otp}\\n\\n"
        "It expires in 5 minutes and can be used only once. "
        "Ignore this email if you did not request a reset."
    )

    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as server:
            server.login(
                os.environ["EMAIL_ADDRESS"],
                os.environ["EMAIL_PASSWORD"]
            )
            server.send_message(message)
        return True
    except Exception:
        with get_db_connection() as conn:
            conn.execute("""
                UPDATE otp_codes SET used = 1
                WHERE email = ? COLLATE NOCASE
                  AND purpose = 'password_reset' AND used = 0
            """, (email,))
        return False
'''

replacements["verify_otp_and_reset_password"] = '''
def verify_otp_and_reset_password(email, otp, new_password, confirmation):
    email = (email or "").strip().lower()
    otp = (otp or "").strip()

    if not valid_email(email):
        return False, "Enter a valid email address."
    if not re.fullmatch(r"\\d{6}", otp):
        return False, "Enter the six-digit OTP."
    if not valid_password(new_password):
        return False, (
            "Password needs 8+ characters, uppercase and lowercase letters, "
            "a digit, and a special character; maximum 72 UTF-8 bytes."
        )
    if new_password != confirmation:
        return False, "Passwords do not match."

    now = datetime.now(timezone.utc)

    with get_db_connection() as conn:
        record = conn.execute("""
            SELECT id, otp_hash, expires_at, attempts
            FROM otp_codes
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset' AND used = 0
            ORDER BY id DESC LIMIT 1
        """, (email,)).fetchone()

        if record is None:
            return False, "OTP is invalid, expired, or already used."

        if record["attempts"] >= 5:
            conn.execute(
                "UPDATE otp_codes SET used = 1 WHERE id = ?",
                (record["id"],)
            )
            return False, "Too many OTP attempts. Request a new code."

        if datetime.fromisoformat(record["expires_at"]) <= now:
            conn.execute(
                "UPDATE otp_codes SET used = 1 WHERE id = ?",
                (record["id"],)
            )
            return False, "OTP has expired. Request a new code."

        conn.execute(
            "UPDATE otp_codes SET attempts = attempts + 1 WHERE id = ?",
            (record["id"],)
        )

        if not verify_password(otp, record["otp_hash"]):
            if record["attempts"] + 1 >= 5:
                conn.execute(
                    "UPDATE otp_codes SET used = 1 WHERE id = ?",
                    (record["id"],)
                )
                return False, "Too many OTP attempts. Request a new code."
            return False, "Incorrect OTP."

        user = conn.execute(
            "SELECT id FROM users WHERE email = ? COLLATE NOCASE",
            (email,)
        ).fetchone()

        if user is None:
            conn.execute(
                "UPDATE otp_codes SET used = 1 WHERE id = ?",
                (record["id"],)
            )
            return False, "Password reset could not be completed."

        # Increment token_version so every previously issued JWT for this
        # account becomes invalid immediately.
        conn.execute("""
            UPDATE users
            SET password_hash = ?, token_version = token_version + 1
            WHERE id = ?
        """, (hash_password(new_password), user["id"]))

        conn.execute("""
            UPDATE otp_codes SET used = 1
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset' AND used = 0
        """, (email,))

    return True, "Password reset successful. You can now log in."
'''

# Replace complete function definitions, working from bottom to top
# so earlier source line numbers remain valid.
lines = original.splitlines(keepends=True)
for name, node in sorted(
    nodes.items(), key=lambda item: item[1].lineno, reverse=True
):
    start = node.lineno - 1
    end = node.end_lineno
    replacement = replacements[name].strip() + "\n\n"
    lines[start:end] = [replacement]

updated = "".join(lines)

# Validate before touching the original file.
ast.parse(updated)
compile(updated, str(APP_PATH), "exec")

# Back up the original, then write the validated replacement.
shutil.copy2(APP_PATH, BACKUP_PATH)
TEMP_PATH.write_text(updated, encoding="utf-8")
os.replace(TEMP_PATH, APP_PATH)

print("✅ Security patch applied.")
print("✅ Original backup saved.")
print("✅ Python syntax validation passed.")
print("Changes: OTP request limit + database-backed JWT invalidation.")


✅ Security patch applied.
✅ Original backup saved.
✅ Python syntax validation passed.
Changes: OTP request limit + database-backed JWT invalidation.


In [28]:

import ast
import sqlite3
from pathlib import Path

app_path = Path("/content/app.py")
source = app_path.read_text(encoding="utf-8")
tree = ast.parse(source)

required = {
    "init_db",
    "make_jwt",
    "verify_jwt",
    "request_password_reset",
    "verify_otp_and_reset_password",
}
found = {
    node.name for node in tree.body
    if isinstance(node, ast.FunctionDef)
}
assert required.issubset(found), "One or more required functions are missing."
assert "token_version" in source, "JWT version control not found."
assert "recent >= 3" in source, "OTP request limit not found."

# Ensure the current database has the new column and index.
with sqlite3.connect("/content/infosys_portal.db") as conn:
    columns = {
        row[1] for row in conn.execute("PRAGMA table_info(users)").fetchall()
    }
    indexes = {
        row[0] for row in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='index'"
        ).fetchall()
    }

print("Functions present:", len(required))
print("Token-version column in database:", "token_version" in columns)
print("OTP request index exists:",
      "idx_otp_email_purpose_created" in indexes)
print("Syntax: OK")


Functions present: 5
Token-version column in database: False
OTP request index exists: False
Syntax: OK


In [29]:

import sqlite3
from pathlib import Path

DB_PATH = "/content/infosys_portal.db"

if not Path(DB_PATH).exists():
    raise FileNotFoundError("Database not found. No changes made.")

with sqlite3.connect(DB_PATH, timeout=15) as conn:
    conn.execute("PRAGMA busy_timeout = 15000")

    # Add the token-version column only if it does not exist.
    columns = {
        row[1]
        for row in conn.execute("PRAGMA table_info(users)").fetchall()
    }

    if "token_version" not in columns:
        conn.execute("""
            ALTER TABLE users
            ADD COLUMN token_version INTEGER NOT NULL DEFAULT 0
        """)
        print("Added token_version column.")
    else:
        print("token_version column already exists.")

    # Add the index only if it does not exist.
    conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_otp_email_purpose_created
        ON otp_codes(email, purpose, created_at)
    """)

    conn.commit()

# Verify the committed schema.
with sqlite3.connect(DB_PATH) as conn:
    columns = {
        row[1]
        for row in conn.execute("PRAGMA table_info(users)").fetchall()
    }
    indexes = {
        row[0]
        for row in conn.execute("""
            SELECT name FROM sqlite_master WHERE type = 'index'
        """).fetchall()
    }

assert "token_version" in columns
assert "idx_otp_email_purpose_created" in indexes

print("\n✅ Database migration verified.")
print("Token-version column:", "token_version" in columns)
print("OTP index:", "idx_otp_email_purpose_created" in indexes)


Added token_version column.

✅ Database migration verified.
Token-version column: True
OTP index: True


In [30]:

import ast
import os
import sqlite3
import tempfile
from contextlib import contextmanager
from datetime import datetime, timezone, timedelta

import jwt

APP_PATH = "/content/app.py"

with open(APP_PATH, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)
wanted = {"make_jwt", "verify_jwt"}
selected = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name in wanted
]

if {node.name for node in selected} != wanted:
    raise RuntimeError("Required JWT functions are missing from app.py.")

# Use a separate temporary database for this test.
fd, test_db_path = tempfile.mkstemp(suffix="_jwt_test.db")
os.close(fd)

@contextmanager
def test_db_connection():
    conn = sqlite3.connect(test_db_path)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

test_globals = {
    "get_db_connection": test_db_connection,
    "jwt": jwt,
    "JWT_SECRET": "temporary-test-secret-not-for-production",
    "JWT_ALGORITHM": "HS256",
    "JWT_EXPIRY_MINUTES": 30,
    "datetime": datetime,
    "timezone": timezone,
    "timedelta": timedelta,
}

try:
    with sqlite3.connect(test_db_path) as conn:
        conn.execute("""
            CREATE TABLE users (
                id INTEGER PRIMARY KEY,
                username TEXT NOT NULL,
                email TEXT NOT NULL,
                token_version INTEGER NOT NULL DEFAULT 0
            )
        """)
        conn.execute("""
            INSERT INTO users (id, username, email, token_version)
            VALUES (1, 'jwt_test_user', 'jwt-test@example.invalid', 0)
        """)

    # Load the actual two functions from app.py into the isolated test setup.
    module = ast.Module(body=selected, type_ignores=[])
    exec(compile(module, APP_PATH, "exec"), test_globals)

    make_jwt = test_globals["make_jwt"]
    verify_jwt = test_globals["verify_jwt"]

    test_user = {
        "id": 1,
        "username": "jwt_test_user",
        "email": "jwt-test@example.invalid",
    }

    # Test 1: a freshly issued token should be valid.
    old_token = make_jwt(test_user)
    assert verify_jwt(old_token) is not None
    print("PASS: Newly issued token is accepted.")

    # Simulate a successful password reset.
    with sqlite3.connect(test_db_path) as conn:
        conn.execute("""
            UPDATE users SET token_version = token_version + 1
            WHERE id = 1
        """)

    # Test 2: the old token must now be rejected.
    assert verify_jwt(old_token) is None
    print("PASS: Old token is rejected after token-version increment.")

    # Test 3: a newly issued token should work with the new version.
    new_token = make_jwt(test_user)
    assert verify_jwt(new_token) is not None
    print("PASS: New token is accepted after version increment.")

    print("\n✅ All JWT invalidation tests passed.")

finally:
    try:
        os.remove(test_db_path)
    except OSError:
        pass


PASS: Newly issued token is accepted.
PASS: Old token is rejected after token-version increment.
PASS: New token is accepted after version increment.

✅ All JWT invalidation tests passed.


In [31]:

import sqlite3
import tempfile
import os
from datetime import datetime, timezone, timedelta

fd, test_db_path = tempfile.mkstemp(suffix="_otp_limit_test.db")
os.close(fd)

try:
    with sqlite3.connect(test_db_path) as conn:
        conn.execute("""
            CREATE TABLE otp_codes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                purpose TEXT NOT NULL,
                created_at TEXT NOT NULL,
                used INTEGER NOT NULL DEFAULT 0
            )
        """)

        now = datetime.now(timezone.utc)
        email = "otp-test@example.invalid"

        # Simulate three requests within the last 15 minutes.
        for minutes_ago in (2, 5, 10):
            created_at = (
                now - timedelta(minutes=minutes_ago)
            ).isoformat()

            conn.execute("""
                INSERT INTO otp_codes
                    (email, purpose, created_at, used)
                VALUES (?, ?, ?, 1)
            """, (email, "password_reset", created_at))

        cutoff = (now - timedelta(minutes=15)).isoformat()

        count = conn.execute("""
            SELECT COUNT(*)
            FROM otp_codes
            WHERE email = ?
              AND purpose = 'password_reset'
              AND created_at >= ?
        """, (email, cutoff)).fetchone()[0]

        print("Requests in last 15 minutes:", count)

        assert count == 3
        assert count >= 3  # A fourth request should be blocked.

        print("PASS: Request limit should block the fourth request.")

        # A request older than 15 minutes should not count.
        conn.execute("""
            INSERT INTO otp_codes
                (email, purpose, created_at, used)
            VALUES (?, ?, ?, 1)
        """, (
            email,
            "password_reset",
            (now - timedelta(minutes=20)).isoformat()
        ))

        count_after_old_request = conn.execute("""
            SELECT COUNT(*)
            FROM otp_codes
            WHERE email = ?
              AND purpose = 'password_reset'
              AND created_at >= ?
        """, (email, cutoff)).fetchone()[0]

        assert count_after_old_request == 3
        print("PASS: Request older than 15 minutes is excluded.")

    print("\n✅ OTP throttling query tests passed.")

finally:
    try:
        os.remove(test_db_path)
    except OSError:
        pass


Requests in last 15 minutes: 3
PASS: Request limit should block the fourth request.
PASS: Request older than 15 minutes is excluded.

✅ OTP throttling query tests passed.


In [34]:

from pyngrok import ngrok

# Close old tunnels to avoid duplicate-tunnel conflicts.
ngrok.kill()

# Create a fresh tunnel to Streamlit's port.
tunnel = ngrok.connect(8501, "http")
print("Open your app here:", tunnel.public_url)


Open your app here: https://compacted-justness-spouse.ngrok-free.dev


In [37]:

import ast
import os
import sqlite3
import tempfile
import secrets
import smtplib
from datetime import datetime, timezone, timedelta
from unittest.mock import patch, MagicMock
from email.mime.text import MIMEText
import bcrypt

APP_PATH = "/content/app.py"

# Read the actual functions from the app source.
with open(APP_PATH, "r", encoding="utf-8") as f:
    tree = ast.parse(f.read())

wanted = {
    "valid_email",
    "request_password_reset",
    "hash_password",
}
selected = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name in wanted
]

found = {node.name for node in selected}
if found != wanted:
    raise RuntimeError(
        f"Expected functions {wanted}; found {found}. "
        "No test was run."
    )

# Create an isolated test database.
fd, test_db_path = tempfile.mkstemp(suffix="_otp_integration.db")
os.close(fd)

def test_get_db_connection():
    conn = sqlite3.connect(test_db_path, timeout=10)
    conn.row_factory = sqlite3.Row
    return conn

import re

test_globals = {
    "get_db_connection": test_get_db_connection,
    "secrets": secrets,
    "datetime": datetime,
    "timezone": timezone,
    "timedelta": timedelta,
    "os": os,
    "smtplib": smtplib,
    "MIMEText": MIMEText,
    "re": re,
    "bcrypt": bcrypt,
}

try:
    with sqlite3.connect(test_db_path) as conn:
        conn.execute("""
            CREATE TABLE users (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                username TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                password_hash TEXT NOT NULL,
                security_question TEXT,
                security_answer_hash TEXT,
                created_at TEXT,
                token_version INTEGER NOT NULL DEFAULT 0
            )
        """)
        conn.execute("""
            CREATE TABLE otp_codes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                otp_hash TEXT NOT NULL,
                purpose TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                used INTEGER NOT NULL DEFAULT 0,
                created_at TEXT NOT NULL
            )
        """)
        conn.execute("""
            INSERT INTO users
                (username, email, password_hash, created_at)
            VALUES (?, ?, ?, ?)
        """, (
            "otp_integration_test",
            "otp-test@example.invalid",
            "not-a-real-password-hash",
            datetime.now(timezone.utc).isoformat()
        ))

    # Load the actual function definitions.
    module = ast.Module(body=selected, type_ignores=[])
    exec(compile(module, APP_PATH, "exec"), test_globals)

    request_password_reset = test_globals["request_password_reset"]

    # Dummy credentials are used only because SMTP is mocked.
    os.environ["EMAIL_ADDRESS"] = "test-sender@example.invalid"
    os.environ["EMAIL_PASSWORD"] = "mock-password"

    # Ensure no real SMTP connection or email can occur.
    with patch.object(smtplib, "SMTP_SSL") as mock_smtp:
        mock_server = MagicMock()
        mock_smtp.return_value.__enter__.return_value = mock_server

        results = []
        for attempt in range(1, 5):
            result = request_password_reset(
                "otp-test@example.invalid"
            )
            results.append(result)
            print(f"Request {attempt}: returned {result}")

        print("Mocked SMTP connections:", mock_smtp.call_count)

        # Three requests should send through the mock;
        # the fourth should be blocked by the rate limit.
        assert results[:3] == [True, True, True], (
            "The first three requests did not all succeed. "
            f"Results: {results}"
        )
        assert results[3] is False, (
            "The fourth request was not blocked. "
            f"Results: {results}"
        )
        assert mock_smtp.call_count == 3, (
            "Unexpected SMTP call count; check the rate-limit behavior."
        )

    with sqlite3.connect(test_db_path) as conn:
        saved_count = conn.execute("""
            SELECT COUNT(*)
            FROM otp_codes
            WHERE email = ?
              AND purpose = 'password_reset'
        """, ("otp-test@example.invalid",)).fetchone()[0]

    assert saved_count == 3, (
        f"Expected 3 OTP records, found {saved_count}."
    )

    print("\nPASS: Three requests were accepted.")
    print("PASS: Fourth request was blocked.")
    print("PASS: No real email was sent.")
    print("PASS: Exactly three OTP records were stored.")
    print("\n✅ OTP request integration test passed.")

finally:
    try:
        os.remove(test_db_path)
    except OSError:
        pass


Request 1: returned True
Request 2: returned True
Request 3: returned True
Request 4: returned False
Mocked SMTP connections: 3

PASS: Three requests were accepted.
PASS: Fourth request was blocked.
PASS: No real email was sent.
PASS: Exactly three OTP records were stored.

✅ OTP request integration test passed.


In [39]:

import ast
import os
import re
import sqlite3
import tempfile
import bcrypt
import jwt
from datetime import datetime, timezone, timedelta

APP_PATH = "/content/app.py"

with open(APP_PATH, "r", encoding="utf-8") as f:
    source = f.read()

tree = ast.parse(source)

wanted = {
    "valid_email",
    "valid_password",
    "hash_password",
    "verify_password",
    "verify_otp_and_reset_password",
    "make_jwt",
    "verify_jwt",
}

selected = [
    node for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name in wanted
]

found = {node.name for node in selected}
if found != wanted:
    raise RuntimeError(f"Missing functions: {wanted - found}")

fd, test_db_path = tempfile.mkstemp(suffix="_reset_test.db")
os.close(fd)

def test_get_db_connection():
    conn = sqlite3.connect(test_db_path, timeout=10)
    conn.row_factory = sqlite3.Row
    return conn

test_globals = {
    "get_db_connection": test_get_db_connection,
    "re": re,
    "bcrypt": bcrypt,
    "jwt": jwt,
    "datetime": datetime,
    "timezone": timezone,
    "timedelta": timedelta,
    "JWT_SECRET": "temporary-test-secret-only",
    "JWT_ALGORITHM": "HS256",
    "JWT_EXPIRY_MINUTES": 30,
}

try:
    with sqlite3.connect(test_db_path) as conn:
        conn.execute("""
            CREATE TABLE users (
                id INTEGER PRIMARY KEY,
                username TEXT NOT NULL,
                email TEXT NOT NULL UNIQUE,
                password_hash TEXT NOT NULL,
                token_version INTEGER NOT NULL DEFAULT 0
            )
        """)
        conn.execute("""
            CREATE TABLE otp_codes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                email TEXT NOT NULL,
                otp_hash TEXT NOT NULL,
                purpose TEXT NOT NULL,
                expires_at TEXT NOT NULL,
                attempts INTEGER NOT NULL DEFAULT 0,
                used INTEGER NOT NULL DEFAULT 0,
                created_at TEXT NOT NULL
            )
        """)

    exec(
        compile(ast.Module(body=selected, type_ignores=[]), APP_PATH, "exec"),
        test_globals
    )

    valid_password = test_globals["valid_password"]
    hash_password = test_globals["hash_password"]
    verify_password = test_globals["verify_password"]
    reset_password = test_globals["verify_otp_and_reset_password"]
    make_jwt = test_globals["make_jwt"]
    verify_jwt = test_globals["verify_jwt"]

    now = datetime.now(timezone.utc)
    future = (now + timedelta(minutes=5)).isoformat()
    past = (now - timedelta(minutes=1)).isoformat()
    test_otp = "123456"
    old_password = "OldPass123!"
    new_password = "NewPass456!"

    assert valid_password(new_password)

    # Three isolated test users.
    users = [
        (1, "success_test", "success@example.invalid",
         hash_password(old_password), 0),
        (2, "attempt_test", "attempts@example.invalid",
         hash_password(old_password), 0),
        (3, "expiry_test", "expired@example.invalid",
         hash_password(old_password), 0),
    ]

    with sqlite3.connect(test_db_path) as conn:
        conn.executemany("""
            INSERT INTO users
                (id, username, email, password_hash, token_version)
            VALUES (?, ?, ?, ?, ?)
        """, users)

        conn.executemany("""
            INSERT INTO otp_codes
                (email, otp_hash, purpose, expires_at, attempts, used, created_at)
            VALUES (?, ?, 'password_reset', ?, 0, 0, ?)
        """, [
            ("success@example.invalid", hash_password(test_otp), future, now.isoformat()),
            ("attempts@example.invalid", hash_password(test_otp), future, now.isoformat()),
            ("expired@example.invalid", hash_password(test_otp), past, now.isoformat()),
        ])

    # Test A: old token invalidation after a successful reset.
    user = {
        "id": 1,
        "username": "success_test",
        "email": "success@example.invalid",
    }

    old_token = make_jwt(user)
    assert verify_jwt(old_token) is not None

    success, message = reset_password(
        "success@example.invalid",
        test_otp,
        new_password,
        new_password
    )
    assert success, f"Expected reset success, got: {message}"
    assert verify_password(
        new_password,
        sqlite3.connect(test_db_path).execute(
            "SELECT password_hash FROM users WHERE id=1"
        ).fetchone()[0]
    )
    assert verify_jwt(old_token) is None

    new_token = make_jwt(user)
    assert verify_jwt(new_token) is not None

    with sqlite3.connect(test_db_path) as conn:
        version = conn.execute(
            "SELECT token_version FROM users WHERE id=1"
        ).fetchone()[0]
        used = conn.execute("""
            SELECT used FROM otp_codes
            WHERE email='success@example.invalid'
        """).fetchone()[0]

    assert version == 1 and used == 1
    print("PASS: Successful reset updates password and consumes OTP.")
    print("PASS: Old JWT rejected; newly issued JWT accepted.")

    # Test B: five incorrect OTP attempts lock the OTP.
    for attempt in range(1, 6):
        success, message = reset_password(
            "attempts@example.invalid",
            "999999",
            new_password,
            new_password
        )
        assert not success

    with sqlite3.connect(test_db_path) as conn:
        attempts, used = conn.execute("""
            SELECT attempts, used FROM otp_codes
            WHERE email='attempts@example.invalid'
        """).fetchone()

    assert attempts == 5 and used == 1
    print("PASS: Five incorrect attempts consume the OTP.")

    # Test C: expired OTP is rejected and marked used.
    success, message = reset_password(
        "expired@example.invalid",
        test_otp,
        new_password,
        new_password
    )
    assert not success

    with sqlite3.connect(test_db_path) as conn:
        used = conn.execute("""
            SELECT used FROM otp_codes
            WHERE email='expired@example.invalid'
        """).fetchone()[0]

    assert used == 1
    print("PASS: Expired OTP rejected and marked used.")

    # Test D: password confirmation mismatch is rejected.
    success, message = reset_password(
        "success@example.invalid",
        test_otp,
        new_password,
        "DifferentPass789!"
    )
    assert not success
    print("PASS: Password confirmation mismatch rejected.")

    print("\n✅ All password-reset integration tests passed.")

finally:
    try:
        os.remove(test_db_path)
    except OSError:
        pass


/usr/local/lib/python3.13/dist-packages/jwt/api_jwt.py:149: InsecureKeyLengthWarning: The HMAC key is 26 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  return self._jws.encode(
/usr/local/lib/python3.13/dist-packages/jwt/api_jwt.py:370: InsecureKeyLengthWarning: The HMAC key is 26 bytes long, which is below the minimum recommended length of 32 bytes for SHA256. See RFC 7518 Section 3.2.
  decoded = self.decode_complete(


PASS: Successful reset updates password and consumes OTP.
PASS: Old JWT rejected; newly issued JWT accepted.
PASS: Five incorrect attempts consume the OTP.
PASS: Expired OTP rejected and marked used.
PASS: Password confirmation mismatch rejected.

✅ All password-reset integration tests passed.


In [42]:

from pathlib import Path
import ast
import shutil

APP_PATH = Path("/content/app.py")
BACKUP_PATH = Path("/content/app.py.pre_security_question_patch.bak")

source = APP_PATH.read_text(encoding="utf-8")

if "# SECURITY QUESTION RECOVERY PATCH" in source:
    raise RuntimeError(
        "Security-question patch marker already exists. "
        "No changes made; do not apply the patch twice."
    )

# Locate the existing code sections before making any changes.
email_start = source.find("def valid_email(email):")
if email_start == -1:
    raise RuntimeError("Could not locate valid_email(). No changes made.")

email_end = source.find("\ndef ", email_start + 1)
if email_end == -1:
    raise RuntimeError("Could not locate the end of valid_email(). No changes made.")

otp_marker = "# ---------- Email OTP password recovery ----------"
if otp_marker not in source:
    raise RuntimeError("Could not locate OTP section. No changes made.")

forgot_start = source.find("    with tab_forgot:")
admin_start = source.find("    with tab_admin:", forgot_start + 1)

if forgot_start == -1 or admin_start == -1:
    raise RuntimeError("Could not locate Forgot Password UI boundaries. No changes made.")

# 1. Tighten email validation to match the assignment.
new_email_function = '''def valid_email(email):
    email = (email or "").strip()

    if not re.fullmatch(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\\\.[A-Za-z]{2,}",
        email
    ):
        return False

    local, domain = email.rsplit("@", 1)
    first_domain_label = domain.split(".", 1)[0]
    final_label = domain.rsplit(".", 1)[-1]

    # Require at least two letters in the local part and
    # at least two letters in the domain label before the first dot.
    local_letters = sum(c.isascii() and c.isalpha() for c in local)
    domain_letters = sum(
        c.isascii() and c.isalpha() for c in first_domain_label
    )

    return (
        local_letters >= 2
        and domain_letters >= 2
        and len(final_label) >= 2
    )
'''

# 2. Add the security-question recovery function.
security_function = '''# SECURITY QUESTION RECOVERY PATCH
def reset_password_with_security_answer(
    username, answer, new_password, confirmation
):
    username = (username or "").strip()
    answer = (answer or "").strip().casefold()

    if not username or not answer:
        return False, "Enter your username and security answer."

    if not valid_password(new_password):
        return False, (
            "Password needs 8+ characters, uppercase and lowercase letters, "
            "a digit, and a special character; maximum 72 UTF-8 bytes."
        )

    if new_password != confirmation:
        return False, "Passwords do not match."

    with get_db_connection() as conn:
        user = conn.execute("""
            SELECT id, email, security_answer_hash
            FROM users
            WHERE username = ? COLLATE NOCASE
        """, (username,)).fetchone()

        if user is None or not verify_password(
            answer, user["security_answer_hash"]
        ):
            return False, "Username or security answer is incorrect."

        conn.execute("""
            UPDATE users
            SET password_hash = ?, token_version = token_version + 1
            WHERE id = ?
        """, (hash_password(new_password), user["id"]))

        # Invalidate any outstanding OTPs for this account too.
        conn.execute("""
            UPDATE otp_codes
            SET used = 1
            WHERE email = ? COLLATE NOCASE
              AND purpose = 'password_reset'
              AND used = 0
        """, (user["email"],))

    return True, "Password reset successful. You can now log in."


'''

# 3. Replace the Forgot Password UI, preserving the existing OTP flow.
new_forgot_ui = '''    with tab_forgot:
        st.subheader("Forgot Password")
        st.write("Choose a recovery method.")

        st.markdown("#### Option 1: Security Question")

        with st.form("find_security_question_form"):
            security_username = st.text_input(
                "Registered username",
                key="security_lookup_username"
            )
            find_question = st.form_submit_button(
                "Find Security Question",
                use_container_width=True
            )

        if find_question:
            lookup_name = security_username.strip()

            if not lookup_name:
                st.error("Enter your registered username.")
                st.session_state.pop("security_reset_username", None)
            else:
                with get_db_connection() as conn:
                    question_row = conn.execute("""
                        SELECT security_question
                        FROM users
                        WHERE username = ? COLLATE NOCASE
                    """, (lookup_name,)).fetchone()

                if question_row:
                    st.session_state["security_reset_username"] = lookup_name
                else:
                    st.session_state.pop("security_reset_username", None)
                    st.error("No matching security question was found.")

        saved_username = st.session_state.get("security_reset_username")

        if saved_username:
            with get_db_connection() as conn:
                question_row = conn.execute("""
                    SELECT security_question
                    FROM users
                    WHERE username = ? COLLATE NOCASE
                """, (saved_username,)).fetchone()

            if question_row:
                st.info("Security question: " + question_row["security_question"])

                with st.form("security_question_reset_form"):
                    security_answer = st.text_input(
                        "Your security answer", type="password"
                    )
                    security_new_password = st.text_input(
                        "New password", type="password"
                    )
                    security_confirm_password = st.text_input(
                        "Confirm new password", type="password"
                    )
                    security_reset_submitted = st.form_submit_button(
                        "Verify Answer and Reset Password",
                        use_container_width=True
                    )

                if security_reset_submitted:
                    ok, message = reset_password_with_security_answer(
                        saved_username,
                        security_answer,
                        security_new_password,
                        security_confirm_password
                    )

                    if ok:
                        st.success(message)
                        st.session_state.pop("security_reset_username", None)
                    else:
                        st.error(message)

        st.divider()
        st.markdown("#### Option 2: Email OTP")
        st.write("Request a six-digit verification code sent to your email.")

        with st.form("request_otp_form"):
            reset_email_input = st.text_input("Registered email address")
            request_otp = st.form_submit_button(
                "Send OTP", use_container_width=True
            )

        if request_otp:
            if not valid_email(reset_email_input):
                st.error("Enter a valid email address.")
            else:
                sent = request_password_reset(reset_email_input)
                st.session_state["reset_email"] = (
                    reset_email_input.strip().lower()
                )
                if sent:
                    st.success(
                        "If this address is registered and email delivery "
                        "succeeded, a code has been sent. Check your inbox."
                    )
                else:
                    st.info(
                        "If the address is registered, check your inbox. "
                        "If no code arrives, verify the email configuration "
                        "and address, then try again."
                    )

        if st.session_state.get("reset_email"):
            st.caption("The OTP expires after 5 minutes.")

            with st.form("verify_otp_form"):
                entered_otp = st.text_input("Six-digit OTP", max_chars=6)
                new_password = st.text_input(
                    "New password", type="password"
                )
                confirm_new_password = st.text_input(
                    "Confirm new password", type="password"
                )
                reset_submitted = st.form_submit_button(
                    "Verify OTP and Reset Password",
                    use_container_width=True
                )

            if reset_submitted:
                ok, message = verify_otp_and_reset_password(
                    st.session_state["reset_email"],
                    entered_otp,
                    new_password,
                    confirm_new_password
                )
                if ok:
                    st.success(message)
                    st.session_state.pop("reset_email", None)
                else:
                    st.error(message)

'''

# Build the proposed source in memory first.
patched = (
    source[:email_start]
    + new_email_function
    + source[email_end:]
)

# Recompute UI boundaries after changing the earlier function.
forgot_start = patched.find("    with tab_forgot:")
admin_start = patched.find("    with tab_admin:", forgot_start + 1)

if forgot_start == -1 or admin_start == -1:
    raise RuntimeError("Could not rebuild UI boundaries. Original file unchanged.")

patched = (
    patched[:forgot_start]
    + new_forgot_ui
    + patched[admin_start:]
)

# Insert the helper before the OTP section.
patched = patched.replace(
    otp_marker,
    security_function + otp_marker,
    1
)

# Validate syntax before writing anything.
ast.parse(patched)

# Preserve the original source and then write the patch.
if not BACKUP_PATH.exists():
    shutil.copy2(APP_PATH, BACKUP_PATH)

APP_PATH.write_text(patched, encoding="utf-8")

# Read back and verify.
saved = APP_PATH.read_text(encoding="utf-8")
ast.parse(saved)

assert "def reset_password_with_security_answer(" in saved
assert "Option 1: Security Question" in saved
assert "Option 2: Email OTP" in saved
assert "def verify_otp_and_reset_password(" in saved

print("✅ Patch applied and Python syntax is valid.")
print("✅ Original backup:", BACKUP_PATH)
print("✅ Security-question recovery added.")
print("✅ Existing OTP recovery function preserved.")
print("✅ Email validation updated.")


✅ Patch applied and Python syntax is valid.
✅ Original backup: /content/app.py.pre_security_question_patch.bak
✅ Security-question recovery added.
✅ Existing OTP recovery function preserved.
✅ Email validation updated.


In [45]:

import ast
import sqlite3
import tempfile
from pathlib import Path
from contextlib import contextmanager

import bcrypt
import jwt

# Read the patched application.
app_source = Path("/content/app.py").read_text(encoding="utf-8")
ast.parse(app_source)

# Extract the actual functions we're testing.
tree = ast.parse(app_source)
wanted = {
    "reset_password_with_security_answer",
    "valid_password",
    "hash_password",
    "verify_password",
    "make_jwt",
    "verify_jwt",
}
selected = [
    node for node in tree.body
    if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef))
    and node.name in wanted
]

found = {node.name for node in selected}
missing = wanted - found
if missing:
    raise RuntimeError(f"Missing functions: {sorted(missing)}")

# Temporary database only.
temp_db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
temp_db.close()

@contextmanager
def get_db_connection():
    conn = sqlite3.connect(temp_db.name)
    conn.row_factory = sqlite3.Row
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

with get_db_connection() as conn:
    conn.execute("""
        CREATE TABLE users (
            id INTEGER PRIMARY KEY,
            username TEXT UNIQUE COLLATE NOCASE,
            email TEXT UNIQUE COLLATE NOCASE,
            password_hash TEXT NOT NULL,
            security_question TEXT NOT NULL,
            security_answer_hash TEXT NOT NULL,
            created_at TEXT NOT NULL,
            token_version INTEGER NOT NULL DEFAULT 0
        )
    """)
    conn.execute("""
        CREATE TABLE otp_codes (
            id INTEGER PRIMARY KEY,
            email TEXT NOT NULL,
            purpose TEXT NOT NULL,
            used INTEGER NOT NULL DEFAULT 0
        )
    """)


import datetime as datetime_module
from datetime import datetime, timezone, timedelta

test_namespace = {
    "re": __import__("re"),
    "hmac": __import__("hmac"),
    "hashlib": __import__("hashlib"),
    "datetime": datetime,
    "timezone": timezone,
    "timedelta": timedelta,
    "jwt": jwt,
    "bcrypt": bcrypt,
    "get_db_connection": get_db_connection,
    "JWT_SECRET": "temporary-test-secret-key-at-least-32-bytes",
    "JWT_ALGORITHM": "HS256",
    "JWT_EXPIRY_MINUTES": 30,
}

# Include any other simple globals needed by these functions.
for name in ("JWT_ALGORITHM", "JWT_SECRET"):
    if name in app_source:
        pass

exec(compile(ast.Module(body=selected, type_ignores=[]),
             "/tmp/security_test_functions.py", "exec"),
     test_namespace)

hash_password = test_namespace["hash_password"]
verify_password = test_namespace["verify_password"]
valid_password = test_namespace["valid_password"]
make_jwt = test_namespace["make_jwt"]
verify_jwt = test_namespace["verify_jwt"]
reset_password = test_namespace["reset_password_with_security_answer"]

old_hash = hash_password("OldPass!123")
answer_hash = hash_password("bluebird")

with get_db_connection() as conn:
    conn.execute("""
        INSERT INTO users (
            id, username, email, password_hash,
            security_question, security_answer_hash,
            created_at, token_version
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        1, "test_user", "test@example.com", old_hash,
        "What is your favourite bird?", answer_hash,
        "2026-01-01", 0
    ))

user = {
    "id": 1,
    "username": "test_user",
    "email": "test@example.com",
}

old_token = make_jwt(user)
assert verify_jwt(old_token), "Newly issued token should be valid"
print("PASS: A newly issued JWT is valid.")

ok, message = reset_password(
    "test_user", "wrong-answer", "NewPass!456", "NewPass!456"
)
assert not ok, "Wrong security answer must fail"
print("PASS: Incorrect security answer rejected.")

ok, message = reset_password(
    "test_user", "bluebird", "NewPass!456", "Different!456"
)
assert not ok, "Password confirmation mismatch must fail"
print("PASS: Password confirmation mismatch rejected.")

ok, message = reset_password(
    "test_user", "bluebird", "NewPass!456", "NewPass!456"
)
assert ok, message
print("PASS: Correct answer resets password.")

with get_db_connection() as conn:
    row = conn.execute(
        "SELECT password_hash, token_version FROM users WHERE id = 1"
    ).fetchone()

assert verify_password("NewPass!456", row["password_hash"])
assert row["token_version"] == 1
assert not verify_jwt(old_token), "Old JWT should be invalidated"
print("PASS: New password works and old JWT is invalidated.")

print("\nAll security-question integration tests passed.")


PASS: A newly issued JWT is valid.
PASS: Incorrect security answer rejected.
PASS: Password confirmation mismatch rejected.
PASS: Correct answer resets password.
PASS: New password works and old JWT is invalidated.

All security-question integration tests passed.


In [49]:

import ast
import re
from pathlib import Path

source = Path("/content/app.py").read_text(encoding="utf-8")
tree = ast.parse(source)

email_node = next(
    node for node in tree.body
    if isinstance(node, ast.FunctionDef) and node.name == "valid_email"
)

namespace = {"re": re}
exec(
    compile(ast.Module(body=[email_node], type_ignores=[]),
            "<email_validation_test>", "exec"),
    namespace
)
valid_email = namespace["valid_email"]

tests = [
    ("krishna@example.com", True),
    ("ab@cd.ef", True),
    ("a@example.com", False),     # Only one letter before @
    ("ab@c.com", False),          # Only one letter before dot
    ("ab@cd.c", False),           # Only one letter after final dot
    ("not-an-email", False),
    ("", False),
]

for email, expected in tests:
    actual = valid_email(email)
    assert actual == expected, (
        f"FAIL: {email!r}: expected {expected}, got {actual}"
    )
    print(f"PASS: {email!r} -> {actual}")

print("\nAll email-validation tests passed.")


PASS: 'krishna@example.com' -> True
PASS: 'ab@cd.ef' -> True
PASS: 'a@example.com' -> False
PASS: 'ab@c.com' -> False
PASS: 'ab@cd.c' -> False
PASS: 'not-an-email' -> False
PASS: '' -> False

All email-validation tests passed.


In [47]:
from pathlib import Path

source = Path("/content/app.py").read_text(encoding="utf-8")
start = source.index("def valid_email(email):")
end = source.index("\ndef ", start + 1)

print(source[start:end])


def valid_email(email):
    email = (email or "").strip()

    if not re.fullmatch(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}",
        email
    ):
        return False

    local, domain = email.rsplit("@", 1)
    first_domain_label = domain.split(".", 1)[0]
    final_label = domain.rsplit(".", 1)[-1]

    # Require at least two letters in the local part and
    # at least two letters in the domain label before the first dot.
    local_letters = sum(c.isascii() and c.isalpha() for c in local)
    domain_letters = sum(
        c.isascii() and c.isalpha() for c in first_domain_label
    )

    return (
        local_letters >= 2
        and domain_letters >= 2
        and len(final_label) >= 2
    )



In [48]:

from pathlib import Path
import ast

app_path = Path("/content/app.py")
source = app_path.read_text(encoding="utf-8")

start = source.index("def valid_email(email):")
end = source.index("\ndef ", start + 1)

correct_function = r'''def valid_email(email):
    email = (email or "").strip()

    if not re.fullmatch(
        r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
        email
    ):
        return False

    local, domain = email.rsplit("@", 1)
    first_domain_label = domain.split(".", 1)[0]
    final_label = domain.rsplit(".", 1)[-1]

    local_letters = sum(
        c.isascii() and c.isalpha() for c in local
    )
    domain_letters = sum(
        c.isascii() and c.isalpha() for c in first_domain_label
    )

    return (
        local_letters >= 2
        and domain_letters >= 2
        and len(final_label) >= 2
    )
'''

updated_source = source[:start] + correct_function + source[end:]
ast.parse(updated_source)
app_path.write_text(updated_source, encoding="utf-8")

print("✅ Email validator corrected.")
print("✅ Python syntax is valid.")


✅ Email validator corrected.
✅ Python syntax is valid.
